In [ ]:
# ============================================================
# RAG PDF Chunk Builder - Full Version
# 목적:
#   PDF 교범을 RAG용 청크 CSV로 변환
#
# 산출물:
#   1. all_rag_chunks.csv
#   2. chunk_metadata.csv
#   3. chunk_build_report.csv
#   4. rag_chunks_package.zip
#
# 사용 흐름:
#   1. Google Drive에 PDF 저장
#   2. PDF_INPUT_DIR 경로 수정
#   3. OUTPUT_DIR 경로 수정
#   4. 전체 실행
# ============================================================

# ------------------------------------------------------------
# 0. 패키지 설치
# ------------------------------------------------------------
!pip -q install pymupdf pandas tqdm

# ------------------------------------------------------------
# 1. 기본 라이브러리
# ------------------------------------------------------------
import os
import re
import json
import zipfile
import hashlib
from pathlib import Path
from datetime import datetime

import fitz  # PyMuPDF
import pandas as pd
from tqdm import tqdm


# ------------------------------------------------------------
# 2. Google Drive 연결
# ------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')


# ------------------------------------------------------------
# 3. 경로 설정
# ------------------------------------------------------------
# 중요:
# 아래 경로는 네 Google Drive 구조에 맞게 수정해야 한다.

PDF_INPUT_DIR = Path("/content/drive/MyDrive/Army_RAG/01_raw_pdf")
OUTPUT_DIR = Path("/content/drive/MyDrive/Army_RAG/02_chunks")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALL_CHUNKS_CSV = OUTPUT_DIR / "all_rag_chunks.csv"
METADATA_CSV = OUTPUT_DIR / "chunk_metadata.csv"
REPORT_CSV = OUTPUT_DIR / "chunk_build_report.csv"
PACKAGE_ZIP = OUTPUT_DIR / "rag_chunks_package.zip"


# ------------------------------------------------------------
# 4. 청킹 파라미터
# ------------------------------------------------------------
# chunk_size:
#   너무 작으면 검색은 잘 되지만 문맥이 부족함
#   너무 크면 검색 정확도가 떨어질 수 있음
#
# overlap_size:
#   앞뒤 청크 사이에 겹치는 텍스트
#   문맥 단절을 줄이기 위한 장치

CHUNK_SIZE = 1200
OVERLAP_SIZE = 200
MIN_CHUNK_SIZE = 250

# 목차, 약어표, 참고문헌 등 제거를 너무 강하게 하면 중요한 내용이 사라질 수 있음.
# 처음에는 False로 두고, 검증 후 필요하면 True로 바꾸는 것을 권장.
FILTER_LOW_VALUE_SECTIONS = True


# ------------------------------------------------------------
# 5. 파일명 기반 문서 ID 정리 함수
# ------------------------------------------------------------
def normalize_doc_id(filename: str) -> str:
    """
    PDF 파일명을 안정적인 문서 ID로 변환한다.
    예:
      FM_3-0_Operations.pdf -> FM_3_0_OPERATIONS
    """
    stem = Path(filename).stem
    stem = stem.replace(" ", "_")
    stem = re.sub(r"[^A-Za-z0-9가-힣_\-\.]", "_", stem)
    stem = stem.replace(".", "_")
    stem = stem.replace("-", "_")
    stem = re.sub(r"_+", "_", stem)
    return stem.upper().strip("_")


# ------------------------------------------------------------
# 6. 텍스트 정제 함수
# ------------------------------------------------------------
def clean_text(text: str) -> str:
    """
    PDF에서 추출한 텍스트를 RAG에 적합하게 정제한다.
    """
    if not text:
        return ""

    # 특수 공백 정리
    text = text.replace("\u00a0", " ")
    text = text.replace("\uf0b7", "-")
    text = text.replace("\u2022", "-")
    text = text.replace("\u2013", "-")
    text = text.replace("\u2014", "-")
    text = text.replace("\r", "\n")

    # 줄 끝 하이픈 연결 제거
    # 예: opera-\ntions -> operations
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # 과도한 공백 정리
    text = re.sub(r"[ \t]+", " ", text)

    # 빈 줄 정리
    text = re.sub(r"\n{3,}", "\n\n", text)

    # 페이지 머리말/꼬리말에서 자주 나오는 단독 페이지 숫자 제거
    lines = []
    for line in text.splitlines():
        line = line.strip()

        if not line:
            lines.append("")
            continue

        # 단독 페이지 번호 제거
        if re.fullmatch(r"\d+", line):
            continue

        # 과도하게 짧은 반복성 라인 일부 제거
        if line.lower() in ["this page intentionally left blank", "intentionally blank"]:
            continue

        lines.append(line)

    text = "\n".join(lines)

    # 다시 빈 줄 정리
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    return text


# ------------------------------------------------------------
# 7. 낮은 가치 섹션 필터링 함수
# ------------------------------------------------------------
def is_low_value_page(text: str) -> bool:
    """
    목차, 참고문헌, 약어표, 빈 페이지 등 RAG 답변 품질을 떨어뜨릴 수 있는 페이지를 식별한다.
    단, 너무 강하게 제거하면 필요한 근거가 빠질 수 있으므로 보수적으로 적용한다.
    """
    if not text or len(text.strip()) < 100:
        return True

    lower = text.lower()

    low_value_patterns = [
        "table of contents",
        "contents",
        "references",
        "bibliography",
        "glossary",
        "acronyms",
        "abbreviations",
        "terms and definitions",
        "index",
    ]

    # 페이지 대부분이 목차성 점선이면 제거
    dot_leader_count = len(re.findall(r"\.{5,}", text))
    if dot_leader_count >= 5:
        return True

    # 명백한 목차/참고문헌/약어표 페이지
    for pat in low_value_patterns:
        if pat in lower[:500]:
            return True

    return False


# ------------------------------------------------------------
# 8. 문단 분리 함수
# ------------------------------------------------------------
def split_into_paragraphs(text: str) -> list:
    """
    텍스트를 문단 단위로 분리한다.
    PDF 추출 특성상 줄바꿈이 불안정하므로,
    빈 줄 기준 + 문장 단위 보완을 함께 사용한다.
    """
    if not text:
        return []

    # 빈 줄 기준 1차 분리
    raw_parts = re.split(r"\n\s*\n", text)

    paragraphs = []
    for part in raw_parts:
        part = part.strip()
        if not part:
            continue

        # 줄바꿈을 문장 내부 공백으로 정리
        part = re.sub(r"\n+", " ", part)
        part = re.sub(r"\s+", " ", part).strip()

        if len(part) < 20:
            continue

        paragraphs.append(part)

    return paragraphs


# ------------------------------------------------------------
# 9. 긴 문단 분할 함수
# ------------------------------------------------------------
def split_long_text(text: str, max_size: int) -> list:
    """
    한 문단이 너무 길 경우 문장 단위로 나눈다.
    """
    if len(text) <= max_size:
        return [text]

    # 영어 문장 기준 분리
    sentences = re.split(r"(?<=[.!?])\s+", text)

    chunks = []
    current = ""

    for sent in sentences:
        sent = sent.strip()
        if not sent:
            continue

        if len(current) + len(sent) + 1 <= max_size:
            current = (current + " " + sent).strip()
        else:
            if current:
                chunks.append(current)
            current = sent

    if current:
        chunks.append(current)

    # 그래도 너무 긴 경우 강제 분할
    final_chunks = []
    for c in chunks:
        if len(c) <= max_size:
            final_chunks.append(c)
        else:
            for i in range(0, len(c), max_size):
                final_chunks.append(c[i:i+max_size])

    return final_chunks


# ------------------------------------------------------------
# 10. 오버랩 생성 함수
# ------------------------------------------------------------
def get_overlap_text(text: str, overlap_size: int) -> str:
    """
    이전 청크의 마지막 일부를 다음 청크 앞에 붙이기 위한 함수.
    """
    if not text:
        return ""

    if len(text) <= overlap_size:
        return text

    return text[-overlap_size:]


# ------------------------------------------------------------
# 11. 페이지 텍스트 추출
# ------------------------------------------------------------
def extract_pdf_pages(pdf_path: Path) -> list:
    """
    PDF에서 페이지별 텍스트를 추출한다.
    반환:
      [
        {
          "page_number": 1,
          "text": "...",
          "char_count": 1234
        },
        ...
      ]
    """
    pages = []

    doc = fitz.open(str(pdf_path))

    for i in range(len(doc)):
        page = doc[i]
        raw_text = page.get_text("text")
        text = clean_text(raw_text)

        pages.append({
            "page_number": i + 1,
            "text": text,
            "char_count": len(text)
        })

    doc.close()
    return pages


# ------------------------------------------------------------
# 12. PDF 하나를 청크로 변환
# ------------------------------------------------------------
def chunk_pdf(pdf_path: Path) -> tuple:
    """
    PDF 하나를 청크 리스트와 리포트로 변환한다.
    """
    doc_id = normalize_doc_id(pdf_path.name)
    doc_title = pdf_path.stem

    pages = extract_pdf_pages(pdf_path)

    chunks = []
    skipped_pages = 0
    used_pages = 0

    chunk_index = 0
    current_text = ""
    current_start_page = None
    current_end_page = None

    for page in pages:
        page_no = page["page_number"]
        page_text = page["text"]

        if FILTER_LOW_VALUE_SECTIONS and is_low_value_page(page_text):
            skipped_pages += 1
            continue

        paragraphs = split_into_paragraphs(page_text)

        if not paragraphs:
            skipped_pages += 1
            continue

        used_pages += 1

        for para in paragraphs:
            # 너무 긴 문단은 먼저 분할
            para_parts = split_long_text(para, CHUNK_SIZE)

            for para_part in para_parts:
                if current_start_page is None:
                    current_start_page = page_no

                candidate = (current_text + "\n\n" + para_part).strip() if current_text else para_part

                if len(candidate) <= CHUNK_SIZE:
                    current_text = candidate
                    current_end_page = page_no
                else:
                    # 현재 청크 저장
                    if len(current_text) >= MIN_CHUNK_SIZE:
                        chunk_index += 1

                        chunk_id = f"{doc_id}_CHUNK_{chunk_index:04d}"

                        citation_text = f"{doc_title}, PDF p.{current_start_page}"
                        if current_end_page and current_end_page != current_start_page:
                            citation_text = f"{doc_title}, PDF pp.{current_start_page}-{current_end_page}"

                        chunks.append({
                            "chunk_id": chunk_id,
                            "doc_id": doc_id,
                            "doc_title": doc_title,
                            "source_file": pdf_path.name,
                            "page_start": current_start_page,
                            "page_end": current_end_page,
                            "citation_text": citation_text,
                            "text": current_text,
                            "char_count": len(current_text),
                            "created_at": datetime.now().isoformat(timespec="seconds")
                        })

                    # 오버랩 적용 후 새 청크 시작
                    overlap = get_overlap_text(current_text, OVERLAP_SIZE)
                    if overlap:
                        current_text = (overlap + "\n\n" + para_part).strip()
                    else:
                        current_text = para_part

                    current_start_page = current_end_page if current_end_page else page_no
                    current_end_page = page_no

                    # 오버랩 때문에 새 청크가 너무 길면 강제 정리
                    if len(current_text) > CHUNK_SIZE:
                        current_text = para_part[-CHUNK_SIZE:]
                        current_start_page = page_no
                        current_end_page = page_no

    # 마지막 청크 저장
    if current_text and len(current_text) >= MIN_CHUNK_SIZE:
        chunk_index += 1

        chunk_id = f"{doc_id}_CHUNK_{chunk_index:04d}"

        citation_text = f"{doc_title}, PDF p.{current_start_page}"
        if current_end_page and current_end_page != current_start_page:
            citation_text = f"{doc_title}, PDF pp.{current_start_page}-{current_end_page}"

        chunks.append({
            "chunk_id": chunk_id,
            "doc_id": doc_id,
            "doc_title": doc_title,
            "source_file": pdf_path.name,
            "page_start": current_start_page,
            "page_end": current_end_page,
            "citation_text": citation_text,
            "text": current_text,
            "char_count": len(current_text),
            "created_at": datetime.now().isoformat(timespec="seconds")
        })

    report = {
        "source_file": pdf_path.name,
        "doc_id": doc_id,
        "doc_title": doc_title,
        "total_pages": len(pages),
        "used_pages": used_pages,
        "skipped_pages": skipped_pages,
        "chunk_count": len(chunks),
        "avg_chunk_chars": round(sum(c["char_count"] for c in chunks) / len(chunks), 1) if chunks else 0,
        "min_chunk_chars": min([c["char_count"] for c in chunks], default=0),
        "max_chunk_chars": max([c["char_count"] for c in chunks], default=0)
    }

    return chunks, report


# ------------------------------------------------------------
# 13. 전체 PDF 처리
# ------------------------------------------------------------
def build_all_chunks(pdf_input_dir: Path, output_dir: Path):
    pdf_files = sorted(list(pdf_input_dir.glob("*.pdf")))

    if not pdf_files:
        raise FileNotFoundError(f"PDF 파일이 없습니다: {pdf_input_dir}")

    all_chunks = []
    reports = []

    print(f"PDF 입력 폴더: {pdf_input_dir}")
    print(f"PDF 파일 수: {len(pdf_files)}")

    for pdf_path in tqdm(pdf_files, desc="PDF 청킹 중"):
        try:
            chunks, report = chunk_pdf(pdf_path)
            all_chunks.extend(chunks)
            reports.append(report)
        except Exception as e:
            reports.append({
                "source_file": pdf_path.name,
                "doc_id": normalize_doc_id(pdf_path.name),
                "doc_title": pdf_path.stem,
                "total_pages": None,
                "used_pages": None,
                "skipped_pages": None,
                "chunk_count": 0,
                "avg_chunk_chars": 0,
                "min_chunk_chars": 0,
                "max_chunk_chars": 0,
                "error": str(e)
            })

    chunks_df = pd.DataFrame(all_chunks)
    report_df = pd.DataFrame(reports)

    if chunks_df.empty:
        raise RuntimeError("생성된 청크가 없습니다. PDF 텍스트 추출 상태를 확인하세요.")

    # 해시 생성
    chunks_df["text_hash"] = chunks_df["text"].apply(
        lambda x: hashlib.md5(x.encode("utf-8")).hexdigest()
    )

    # 메타데이터 분리
    metadata_columns = [
        "chunk_id",
        "doc_id",
        "doc_title",
        "source_file",
        "page_start",
        "page_end",
        "citation_text",
        "char_count",
        "text_hash",
        "created_at"
    ]

    metadata_df = chunks_df[metadata_columns].copy()

    # 저장
    chunks_df.to_csv(ALL_CHUNKS_CSV, index=False, encoding="utf-8-sig")
    metadata_df.to_csv(METADATA_CSV, index=False, encoding="utf-8-sig")
    report_df.to_csv(REPORT_CSV, index=False, encoding="utf-8-sig")

    # ZIP 패키징
    with zipfile.ZipFile(PACKAGE_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(ALL_CHUNKS_CSV, arcname=ALL_CHUNKS_CSV.name)
        zf.write(METADATA_CSV, arcname=METADATA_CSV.name)
        zf.write(REPORT_CSV, arcname=REPORT_CSV.name)

    print("\n청킹 완료")
    print(f"전체 청크 수: {len(chunks_df)}")
    print(f"문서 수: {chunks_df['source_file'].nunique()}")
    print(f"평균 청크 길이: {chunks_df['char_count'].mean():.1f}자")
    print(f"저장 위치: {output_dir}")
    print(f"- {ALL_CHUNKS_CSV}")
    print(f"- {METADATA_CSV}")
    print(f"- {REPORT_CSV}")
    print(f"- {PACKAGE_ZIP}")

    return chunks_df, metadata_df, report_df


# ------------------------------------------------------------
# 14. 실행
# ------------------------------------------------------------
chunks_df, metadata_df, report_df = build_all_chunks(PDF_INPUT_DIR, OUTPUT_DIR)


# ------------------------------------------------------------
# 15. 결과 미리보기
# ------------------------------------------------------------
display(report_df)
display(chunks_df.head(5))